# 05 - Final Load Preparation

Objective: Compute business KPIs and create final Tableau-ready data load files.

In [1]:
import os
import numpy as np
import pandas as pd

INPUT_PATH = '../data/processed/blinkit_cleaned_dashboard_data.csv'
OUTPUT_DIR = '../data/processed'

df = pd.read_csv(INPUT_PATH)

for c in ['order_date', 'promised_time', 'actual_time', 'feedback_date']:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors='coerce')

order_level = df.sort_values('order_date').drop_duplicates('order_id').copy()

print(f'Line-item rows: {len(df):,}')
print(f'Order-level rows: {len(order_level):,}')

Line-item rows: 5,000
Order-level rows: 5,000


## KPI Computation

In [2]:
kpis = {
    'total_orders': int(order_level['order_id'].nunique()),
    'total_revenue': float(order_level['order_total'].sum()),
    'avg_order_value': float(order_level['order_total'].mean()),
    'median_order_value': float(order_level['order_total'].median()),
    'on_time_rate_pct': float((~order_level['is_delayed'].fillna(False)).mean() * 100),
    'delay_rate_pct': float(order_level['is_delayed'].fillna(False).mean() * 100),
    'avg_delay_minutes': float(order_level['delay_minutes'].fillna(0).mean()),
    'avg_actual_duration_min': float(order_level['actual_duration_minutes'].mean()),
    'avg_promised_duration_min': float(order_level['promised_duration_minutes'].mean()),
    'avg_rating': float(order_level['rating'].mean()),
    'peak_hour_order_share_pct': float(order_level['is_peak'].fillna(False).mean() * 100),
    'active_customers': int(order_level['customer_id'].nunique())
}

kpi_df = pd.DataFrame(list(kpis.items()), columns=['kpi_name', 'kpi_value'])
display(kpi_df)

,kpi_name,kpi_value
0,total_orders,5.000000e+03
1,total_revenue,1.100931e+07
2,avg_order_value,2.201862e+03
3,median_order_value,2.100690e+03
4,on_time_rate_pct,3.804000e+01
5,delay_rate_pct,6.196000e+01
6,avg_delay_minutes,5.381800e+00
7,avg_actual_duration_min,1.943440e+01
8,avg_promised_duration_min,1.499140e+01
9,avg_rating,3.344400e+00


## Final Dashboard Load (Tableau Public)

In [3]:
# Order-level final table for dashboarding
tableau_orders = order_level[[
    'order_id', 'customer_id', 'order_date', 'weekday', 'month', 'hour',
    'area', 'pincode', 'customer_segment', 'payment_method',
    'order_total', 'order_size', 'order_value_bucket',
    'delivery_status', 'delay_category', 'delay_bucket',
    'distance_km', 'promised_duration_minutes', 'actual_duration_minutes',
    'delivery_efficiency', 'is_delayed', 'delay_minutes',
    'rating', 'sentiment', 'feedback_category', 'is_peak'
]].copy()

tableau_orders['order_date_only'] = tableau_orders['order_date'].dt.date
tableau_orders['order_year'] = tableau_orders['order_date'].dt.year
tableau_orders['order_month_num'] = tableau_orders['order_date'].dt.month
tableau_orders['order_month_name'] = tableau_orders['order_date'].dt.month_name()
tableau_orders['order_year_month'] = tableau_orders['order_date'].dt.to_period('M').astype(str)

# Category summary table from line-item data
tableau_category = df.groupby('category', as_index=False).agg(
    category_revenue=('total_price', 'sum'),
    units_sold=('quantity', 'sum'),
    avg_item_price=('unit_price', 'mean'),
    avg_margin=('margin_percentage', 'mean')
)

# Area summary table from order-level data
tableau_area = order_level.groupby('area', as_index=False).agg(
    total_orders=('order_id', 'nunique'),
    total_revenue=('order_total', 'sum'),
    avg_delivery_minutes=('actual_duration_minutes', 'mean'),
    delay_rate=('is_delayed', 'mean')
)
tableau_area['delay_rate_pct'] = tableau_area['delay_rate'] * 100

display(tableau_orders.head())
display(tableau_category.sort_values('category_revenue', ascending=False).head(10))
display(tableau_area.sort_values('total_revenue', ascending=False).head(10))

,order_id,customer_id,order_date,weekday,month,hour,area,pincode,customer_segment,payment_method,...,delay_minutes,rating,sentiment,feedback_category,is_peak,order_date_only,order_year,order_month_num,order_month_name,order_year_month
575,7164827734,9195970,2023-03-16 08:10:44,Thursday,3,8,Karawal Nagar,785030,New,UPI,...,0.0,3,Neutral,App Experience,True,2023-03-16,2023,3,March,2023-03
3770,4028266143,19216141,2023-03-16 08:31:08,Thursday,3,8,Giridih,829446,Premium,Card,...,2.0,1,Negative,Delivery,True,2023-03-16,2023,3,March,2023-03
1017,4376524554,72576730,2023-03-16 09:09:00,Thursday,3,9,Chennai,734650,Regular,Card,...,5.0,3,Neutral,Customer Service,True,2023-03-16,2023,3,March,2023-03
1199,208989475,94713625,2023-03-16 11:38:32,Thursday,3,11,Barasat,284684,Premium,Cash,...,0.0,5,Positive,Delivery,True,2023-03-16,2023,3,March,2023-03
4770,757816226,22571994,2023-03-16 14:07:42,Thursday,3,14,Bihar Sharif,83717,Regular,Wallet,...,18.0,2,Negative,Customer Service,False,2023-03-16,2023,3,March,2023-03


,category,category_revenue,units_sold,avg_item_price,avg_margin
2,Dairy & Breakfast,639222.19,1114,574.776396,20.0
9,Pharmacy,592368.57,973,601.520021,20.0
3,Fruits & Vegetables,559053.08,966,566.480488,25.0
8,Pet Care,539888.75,1003,535.208583,35.0
5,Household Care,444244.25,1078,409.000511,25.0
7,Personal Care,394894.61,887,438.945947,35.0
10,Snacks & Munchies,394648.71,963,417.271884,35.0
1,Cold Drinks & Juices,392717.62,758,517.329040,30.0
4,Grocery & Staples,359937.82,895,400.935590,15.0
0,Baby Care,348227.18,655,521.659311,30.0


,area,total_orders,total_revenue,avg_delivery_minutes,delay_rate,delay_rate_pct
218,Orai,44,99590.96,21.409091,0.750000,75.000000
80,Deoghar,40,95386.05,18.300000,0.675000,67.500000
207,Nandyal,36,83281.10,18.111111,0.611111,61.111111
97,Gandhinagar,37,82273.95,19.891892,0.675676,67.567568
52,Bhopal,31,78854.32,19.580645,0.612903,61.290323
100,Ghaziabad,32,72599.85,20.968750,0.562500,56.250000
4,Ahmednagar,30,72233.14,17.900000,0.633333,63.333333
126,Jabalpur,30,70135.67,20.233333,0.700000,70.000000
130,Jalna,30,69086.31,17.166667,0.500000,50.000000
248,Ratlam,35,67426.31,18.514286,0.571429,57.142857


In [4]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

kpi_path = os.path.join(OUTPUT_DIR, 'blinkit_kpis.csv')
orders_path = os.path.join(OUTPUT_DIR, 'tableau_final_orders.csv')
category_path = os.path.join(OUTPUT_DIR, 'tableau_final_category_summary.csv')
area_path = os.path.join(OUTPUT_DIR, 'tableau_final_area_summary.csv')

kpi_df.to_csv(kpi_path, index=False)
tableau_orders.to_csv(orders_path, index=False)
tableau_category.to_csv(category_path, index=False)
tableau_area.to_csv(area_path, index=False)

print('Saved:')
print('-', kpi_path)
print('-', orders_path)
print('-', category_path)
print('-', area_path)

Saved:
- ../data/processed/blinkit_kpis.csv
- ../data/processed/tableau_final_orders.csv
- ../data/processed/tableau_final_category_summary.csv
- ../data/processed/tableau_final_area_summary.csv
